# UAS Praktikum NLP — Chatbot Bantuan Hukum Masyarakat Awam
**Mata Kuliah**: Natural Language Processing (NLP)  
**Topik**: Sistem Chatbot berbasis LangChain + LangGraph + LangSmith  
**Lingkungan**: Google Colab  

---

## Latar Belakang Masalah

Indonesia memiliki lebih dari 270 juta penduduk, namun rasio pengacara hanya sekitar 1 berbanding 3.000. Biaya konsultasi hukum yang mahal, bisa lebih dari Rp500.000 per jam, membuat sebagian besar masyarakat tidak mampu mengaksesnya.

Banyak warga tidak tahu hak-hak dasar mereka, misalnya hak pesangon ketika di-PHK, atau hak sebagai konsumen saat membeli produk yang rusak. Ketidaktahuan ini sering berujung pada kerugian yang sebenarnya bisa dihindari jika ada informasi yang mudah diakses.

Sistem yang dibangun pada notebook ini adalah chatbot berbasis RAG (Retrieval Augmented Generation) yang menjawab pertanyaan hukum berdasarkan dokumen Undang-Undang resmi, dalam bahasa yang mudah dipahami masyarakat awam.

---

## Library Wajib yang Digunakan

| Library | Peran dalam Sistem |
|---|---|
| LangChain | RAG pipeline, memuat dan mengindeks dokumen PDF, vector store, prompt template |
| LangGraph | State machine 4 node: clarify, retrieve, analyze, recommend |
| LangSmith | Monitoring, tracing, dan evaluasi setiap langkah pipeline |

---

## Dokumen Hukum yang Digunakan

| Dokumen | Cakupan Topik |
|---|---|
| UU No. 13 Tahun 2003 tentang Ketenagakerjaan | PHK, pesangon, upah, kontrak kerja |
| UU No. 8 Tahun 1999 tentang Perlindungan Konsumen | Hak konsumen, garansi, refund, penipuan produk |

---

## Arsitektur Sistem

```
Pertanyaan dari User
        |
        v
+----------------------------------------------+
|         LangChain (Lapisan Orkestrasi)        |
|                                              |
|   PDF Loader -> Text Splitter -> Embeddings  |
|        +-----------------------------+        |
|                                     v        |
|                                 ChromaDB     |
|                                     |        |
|   +----------------------------------v-----+ |
|   |         LangGraph State Machine        | |
|   |   [1] clarify_context                  | |
|   |        |                               | |
|   |   [2] retrieve_documents               | |
|   |        |                               | |
|   |   [3] analyze_and_answer               | |
|   |        |                               | |
|   |   [4] recommend_action                 | |
|   +----------------------------------------+ |
+----------------------------------------------+
        |                        |
        v                        v
  Respons Final             LangSmith
  (ke User)            (Tracing & Evaluasi)
```


---
# 1. Instalasi Library

Sebelum memulai, kita install semua library yang dibutuhkan. Pada tahap ini kita menyiapkan library yang diperlukan. Setiap library memiliki fungsi yang berbeda sesuai kebutuhan sistem.

- `langchain`, `langchain-openai`, `langchain-community` — framework utama untuk membangun RAG pipeline
- `langgraph` — digunakan untuk membangun alur kerja multi-langkah berbasis state machine
- `langsmith` — untuk monitoring dan tracing setiap langkah yang dijalankan sistem
- `chromadb` — database vektor sebagai tempat penyimpanan dokumen yang sudah diindeks
- `pypdf` — untuk membaca dan mengekstrak teks dari file PDF
- `gradio` — untuk membuat tampilan antarmuka chatbot yang bisa diakses lewat browser

Proses instalasi pertama kali bisa memakan waktu 2 sampai 5 menit. Jika ada pesan warning kecil, itu normal dan tidak menghalangi notebook berjalan.


In [ ]:
%%capture

!pip install langchain langchain-openai langchain-community langgraph langsmith chromadb pypdf gradio tiktoken openai

print("Instalasi selesai!")

---
# 2. Import Library

Setelah instalasi selesai, kita impor semua modul yang akan digunakan. Library diimpor berdasarkan fungsinya agar struktur program lebih mudah dipahami. bagian LangChain menangani RAG dan komunikasi dengan model bahasa, bagian LangGraph untuk membangun alur state machine, dan bagian LangSmith untuk monitoring.

Jika ada error saat menjalankan cell ini, coba restart runtime Colab terlebih dahulu, lalu jalankan ulang dari cell instalasi.


In [ ]:
# Import semua library yang akan digunakan
import os
import json
import re
from typing import TypedDict, List, Optional

# LangChain
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# LangGraph
from langgraph.graph import StateGraph, END

# LangSmith
from langsmith import traceable, Client as LangSmithClient

# Utilitas
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA
import numpy as np
import gradio as gr
from pathlib import Path

try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

print("Semua library berhasil diimpor!")
print("LangChain  ✓")
print("LangGraph  ✓")
print("LangSmith  ✓")


/tmp/ipykernel_786/709262170.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


Semua library berhasil diimpor!
LangChain  ✓
LangGraph  ✓
LangSmith  ✓


---
# 3. Setup API Key dan Aktivasi LangSmith

Ada dua API key yang dibutuhkan sistem ini:

1. `OPENAI_API_KEY` — digunakan untuk memanggil model `gpt-4.1-mini` saat menjawab pertanyaan, dan model `text-embedding-3-small` saat mengubah teks menjadi vektor
2. `LANGCHAIN_API_KEY` — digunakan untuk mengaktifkan LangSmith agar setiap langkah pipeline tercatat secara otomatis

Cara menyimpan API key dengan aman di Colab: buka ikon kunci di sidebar kiri, tambahkan nama key beserta nilainya, lalu aktifkan toggle akses notebook. Dengan cara ini API key tidak tertulis langsung di kode.

Variabel `LANGCHAIN_TRACING_V2` yang diset ke `true` adalah yang mengaktifkan pencatatan ke LangSmith. Nama project `uas-chatbot-hukum` akan muncul sebagai label di dashboard LangSmith.


In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("GPT")
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "uas-chatbot-hukum"

print("OpenAI Key :", os.environ["OPENAI_API_KEY"][:20])
print("LangSmith Key :", os.environ["LANGCHAIN_API_KEY"][:15])
print("Setup berhasil")

OpenAI Key : sk-proj-RDcXgwFT9Mkb
LangSmith Key : lsv2_pt_dff4986
Setup berhasil


---
# 4. Inisialisasi Model Bahasa dan Embedding

Di sini kita siapkan dua komponen inti dari LangChain:

- `ChatOpenAI` dengan model `gpt-4.1-mini` — model bahasa yang digunakan untuk memahami pertanyaan dan menyusun jawaban. Parameter `temperature=0` memastikan jawaban yang dihasilkan selalu konsisten dan tidak acak, yang penting untuk konteks hukum yang membutuhkan ketepatan.
- `OpenAIEmbeddings` dengan model `text-embedding-3-small` — model ini mengubah setiap potongan teks menjadi representasi vektor berisi 1536 angka. Vektor inilah yang nantinya dipakai untuk mencari dokumen yang paling relevan dengan pertanyaan user.

Setelah inisialisasi, kita jalankan satu test sederhana untuk memastikan koneksi ke API berjalan dan key yang dimasukkan valid.


In [ ]:
# Inisialisasi model bahasa dan model embedding dari LangChain
# Inisialisasi LLM
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# Inisialisasi Embedding
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Verifikasi model yang aktif
print("Model yang digunakan:")
print(f"  LLM       : {llm.model_name}")
print(f"  Embedding : {embeddings.model}")
print()

# Test invoke singkat
test_response = llm.invoke("Jawab dalam satu kalimat: Apa itu LangChain?")
print("Test LLM berhasil!")
print(f"Respons  : {test_response.content}")


Model yang digunakan:
  LLM       : gpt-4.1-mini
  Embedding : text-embedding-3-small

Test LLM berhasil!
Respons  : LangChain adalah sebuah framework yang memudahkan pengembangan aplikasi berbasis model bahasa besar (LLM) dengan mengintegrasikan berbagai komponen seperti pemrosesan bahasa alami, pengambilan data, dan manajemen alur kerja.


---
# 5. Upload Dokumen Hukum PDF

Pada tahap ini kita upload dua file PDF yang akan dijadikan sumber pengetahuan sistem. Kedua file ini adalah dokumen Undang-Undang resmi yang diunduh dari situs resmi pemerintah.

File yang dibutuhkan:
1. `uu_ketenagakerjaan.pdf` — UU No. 13 Tahun 2003
2. `uu_perlindungan_konsumen.pdf` — UU No. 8 Tahun 1999

Sistem akan mengecek apakah file sudah ada terlebih dahulu. Jika belum, dialog upload akan muncul secara otomatis. Pastikan nama file persis sama dengan yang disebutkan di atas, termasuk huruf kecil dan tanpa spasi, agar sistem bisa mendeteksi domain hukumnya dengan benar.


In [ ]:
# Upload dan verifikasi dokumen hukum PDF
from google.colab import files

FILE_NAMES = [
    "uu_ketenagakerjaan.pdf",
    "uu_perlindungan_konsumen.pdf"
]

DOMAIN_MAP = {
    "uu_ketenagakerjaan"      : "ketenagakerjaan",
    "uu_perlindungan_konsumen": "konsumen",
}

# Cek file yang sudah ada
sudah_ada = [f for f in FILE_NAMES if Path(f).exists()]
belum_ada = [f for f in FILE_NAMES if not Path(f).exists()]

if sudah_ada:
    print("File yang sudah tersedia:")
    for f in sudah_ada:
        size_kb = Path(f).stat().st_size // 1024
        print(f"  ✓ {f} ({size_kb} KB)")

if belum_ada:
    print(f"\nFile berikut belum ditemukan, silakan upload:")
    for f in belum_ada:
        print(f"  - {f}")
    print()
    uploaded = files.upload()
    print("\nFile berhasil diupload:")
    for f in uploaded:
        print(f"  ✓ {f}")

# Verifikasi akhir
print("\nVerifikasi file:")
semua_ada = True
for f in FILE_NAMES:
    if Path(f).exists():
        size_kb = Path(f).stat().st_size // 1024
        print(f"  ✓ {f} ({size_kb} KB)")
    else:
        print(f"  ✗ {f} — TIDAK DITEMUKAN")
        semua_ada = False

if semua_ada:
    print("\nSemua dokumen hukum siap diproses!")
else:
    print("\n⚠️ Ada file yang belum diupload. Upload ulang sebelum lanjut.")



File berikut belum ditemukan, silakan upload:
  - uu_ketenagakerjaan.pdf
  - uu_perlindungan_konsumen.pdf



Saving uu_perlindungan_konsumen.pdf to uu_perlindungan_konsumen.pdf
Saving uu_ketenagakerjaan.pdf to uu_ketenagakerjaan.pdf

File berhasil diupload:
  ✓ uu_perlindungan_konsumen.pdf
  ✓ uu_ketenagakerjaan.pdf

Verifikasi file:
  ✓ uu_ketenagakerjaan.pdf (176 KB)
  ✓ uu_perlindungan_konsumen.pdf (43 KB)

Semua dokumen hukum siap diproses!


---
# 6. LangChain — Memuat dan Mengindeks Dokumen PDF

Inilah bagian utama dari komponen LangChain. Ada empat proses yang berjalan secara berurutan:

**Load** — `PyPDFLoader` membaca setiap halaman dari file PDF dan mengubahnya menjadi objek dokumen yang bisa diproses Python.

**Split** — `RecursiveCharacterTextSplitter` memecah teks yang panjang menjadi potongan-potongan kecil yang disebut chunk. Ukuran setiap chunk dibatasi 800 karakter dengan overlap 150 karakter antar chunk. Kita prioritaskan pemisahan di kata "Pasal" agar setiap chunk sedapat mungkin berisi satu pasal yang utuh dan tidak terpotong di tengah kalimat.

**Embed** — Setiap chunk dikirim ke model `text-embedding-3-small` untuk diubah menjadi vektor angka yang merepresentasikan makna semantiknya. Proses ini memerlukan pemanggilan ke API OpenAI.

**Store** — Semua vektor disimpan ke ChromaDB. Kita juga menyertakan metadata `domain` pada setiap chunk agar saat pencarian nanti bisa disaring berdasarkan jenis dokumennya. Parameter `hnsw:space: cosine` memastikan ChromaDB menggunakan cosine similarity, yang menghasilkan skor dalam rentang yang lebih konsisten.

Jumlah chunk yang dihasilkan bergantung pada tebal dokumen. Semakin banyak pasal, semakin banyak chunk yang terbentuk.


In [ ]:
# Proses Load, Split, Embed, dan Store dokumen PDF ke ChromaDB
# Text splitter — prioritaskan pemisahan per Pasal
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["Pasal ", "\n\n", "\n", ". ", " "]
)

all_chunks = []

print("Memuat dan memproses dokumen PDF...\n")

for filename in FILE_NAMES:
    if not Path(filename).exists():
        print(f"  ✗ {filename} tidak ditemukan, dilewati.")
        continue

    # Deteksi domain dari nama file
    stem   = Path(filename).stem
    domain = DOMAIN_MAP.get(stem, "umum")

    # Load PDF
    loader = PyPDFLoader(filename)
    pages  = loader.load()

    # Split menjadi chunks
    chunks = splitter.split_documents(pages)

    # Tambahkan metadata domain ke setiap chunk
    for chunk in chunks:
        chunk.metadata["domain"] = domain
        chunk.metadata["source"] = filename

    all_chunks.extend(chunks)

    print(f"  ✓ {filename}")
    print(f"    Domain  : {domain}")
    print(f"    Halaman : {len(pages)}")
    print(f"    Chunks  : {len(chunks)}")
    print()

print(f"Total seluruh chunks : {len(all_chunks)}")

# Buat Chroma vector store (LangChain)
print("\nMembuat embeddings dan menyimpan ke ChromaDB...")
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    collection_name="dokumen_hukum_uas"
)

print(f"Vector store berhasil dibuat!")
print(f"Total dokumen terindeks : {vectorstore._collection.count()}")

# Buat retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("Retriever siap digunakan.")


Memuat dan memproses dokumen PDF...

  ✓ uu_ketenagakerjaan.pdf
    Domain  : ketenagakerjaan
    Halaman : 50
    Chunks  : 229

  ✓ uu_perlindungan_konsumen.pdf
    Domain  : konsumen
    Halaman : 16
    Chunks  : 81

Total seluruh chunks : 310

Membuat embeddings dan menyimpan ke ChromaDB...
Vector store berhasil dibuat!
Total dokumen terindeks : 310
Retriever siap digunakan.


---
## 6a. Uji Coba Pencarian Semantik

Sebelum melanjutkan ke pembangunan sistem, kita uji dulu apakah retrieval bekerja dengan benar. Caranya adalah mengirim sebuah pertanyaan uji, lalu melihat chunk-chunk mana yang berhasil ditemukan oleh sistem.

Skor similarity menunjukkan seberapa mirip chunk tersebut dengan pertanyaan yang diajukan. Semakin tinggi nilainya, semakin relevan kontennya. Jika hasil yang muncul sesuai topik pertanyaan, artinya RAG pipeline berjalan dengan benar dan siap digunakan oleh node-node LangGraph.


In [ ]:
# Uji coba pencarian semantik dengan query tentang PHK
query_uji = "Berapa pesangon yang didapat jika di-PHK setelah bekerja 5 tahun?"

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    collection_name="dokumen_hukum_uas",
    collection_metadata={"hnsw:space": "cosine"}  # ← tambahkan ini
)
docs_skor = vectorstore.similarity_search_with_relevance_scores(query_uji, k=3)

print(f"Query: {query_uji}")
print(f"Jumlah dokumen ditemukan: {len(docs_skor)}\n")

for i, (doc, score) in enumerate(docs_skor, 1):
    print(f"[Dokumen {i}]")
    print(f"  Domain  : {doc.metadata.get('domain', '-')}")
    print(f"  Sumber  : {doc.metadata.get('source', '-')}")
    print(f"  Score   : {score:.4f}")
    print(f"  Isi     : {doc.page_content[:250]}...")
    print()


Query: Berapa pesangon yang didapat jika di-PHK setelah bekerja 5 tahun?
Jumlah dokumen ditemukan: 3

[Dokumen 1]
  Domain  : ketenagakerjaan
  Sumber  : uu_ketenagakerjaan.pdf
  Score   : 0.4348
  Isi     : h. masa kerja 24 (dua puluh empat) tahun atau lebih, 10 (sepuluh ) bulan upah.
(4) Uang penggantian hak yang seharusnya diterima sebagaimana dimaksud dalam ayat (1) meliputi :
a. cuti tahunan yang belum diambil dan belum gugur;
b. biaya atau ongkos p...

[Dokumen 2]
  Domain  : ketenagakerjaan
  Sumber  : uu_ketenagakerjaan.pdf
  Score   : 0.4343
  Isi     : h. masa kerja 24 (dua puluh empat) tahun atau lebih, 10 (sepuluh ) bulan upah.
(4) Uang penggantian hak yang seharusnya diterima sebagaimana dimaksud dalam ayat (1) meliputi :
a. cuti tahunan yang belum diambil dan belum gugur;
b. biaya atau ongkos p...

[Dokumen 3]
  Domain  : ketenagakerjaan
  Sumber  : uu_ketenagakerjaan.pdf
  Score   : 0.3929
  Isi     : Pasal 156
(1) Dalam hal terjadi pemutusan hubungan kerja, pengusaha di

---
# 7. LangGraph — Mendefinisikan AgentState

Sebelum membangun node-node LangGraph, kita perlu mendefinisikan `AgentState`. Ini adalah sebuah TypedDict yang berperan sebagai wadah data yang dibawa dari satu node ke node berikutnya selama pipeline berjalan.

Setiap node menerima state yang berisi semua data yang ada saat itu, lalu mengubah atau menambahkan beberapa field sesuai tugasnya, kemudian meneruskan state yang sudah diperbarui ke node selanjutnya.

Perhatikan pengelompokan field berdasarkan node yang mengisinya. Field `user_query` dan `conversation_history` adalah input awal dari user. Field seperti `legal_domain` dan `needs_clarification` diisi oleh node pertama. Field `retrieved_docs` diisi oleh node kedua, dan seterusnya sampai `final_response` yang diisi oleh node terakhir.


In [ ]:
# Definisi AgentState sebagai wadah data antar node LangGraph
class AgentState(TypedDict):
    # Input dari user
    user_query            : str
    conversation_history  : List[dict]

    # Output node 1 — clarify
    legal_domain          : Optional[str]   # ketenagakerjaan / konsumen / lainnya
    clarified_query       : Optional[str]
    needs_clarification   : bool

    # Output node 2 — retrieve
    retrieved_docs        : List[Document]
    source_references     : List[str]
    relevance_scores      : List[float]

    # Output node 3 — analyze
    answer                : Optional[str]
    cited_articles        : List[str]
    confidence_level      : float

    # Output node 4 — recommend
    action_steps          : List[str]
    final_response        : Optional[str]

print("AgentState berhasil didefinisikan.")
print("\nField yang tersedia:")
for field, ftype in AgentState.__annotations__.items():
    print(f"  - {field}: {ftype}")


AgentState berhasil didefinisikan.

Field yang tersedia:
  - user_query: <class 'str'>
  - conversation_history: typing.List[dict]
  - legal_domain: typing.Optional[str]
  - clarified_query: typing.Optional[str]
  - needs_clarification: <class 'bool'>
  - retrieved_docs: typing.List[langchain_core.documents.base.Document]
  - source_references: typing.List[str]
  - relevance_scores: typing.List[float]
  - answer: typing.Optional[str]
  - cited_articles: typing.List[str]
  - confidence_level: <class 'float'>
  - action_steps: typing.List[str]
  - final_response: typing.Optional[str]


---
# 8. LangGraph Node 1 — clarify_context

Node pertama bertugas melakukan dua hal: mengidentifikasi domain hukum dari pertanyaan yang masuk, dan menilai apakah pertanyaan sudah cukup jelas untuk diproses lebih lanjut.

Model bahasa diminta menjawab dalam format JSON dengan tiga informasi: `domain` (ketenagakerjaan, konsumen, atau lainnya), `is_clear` (apakah pertanyaan sudah jelas), dan `clarification_question` (pertanyaan balik jika perlu klarifikasi lebih lanjut).

Decorator `@traceable` di atas fungsi adalah instruksi ke LangSmith untuk mencatat setiap kali node ini dijalankan, termasuk input, output, dan waktu eksekusinya.

Setelah node ini berjalan, ada dua kemungkinan arah yang bisa ditempuh:
- Jika pertanyaan belum jelas, pipeline berhenti dan mengembalikan pertanyaan klarifikasi ke user
- Jika sudah jelas, pipeline lanjut ke node berikutnya untuk mencari dokumen yang relevan


In [ ]:
# Node 1: Identifikasi domain hukum dan validasi pertanyaan user
CLARIFY_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """\
Kamu adalah asisten hukum Indonesia yang mengidentifikasi jenis pertanyaan hukum.

Domain yang tersedia HANYA dua:
- ketenagakerjaan: PHK, pesangon, upah, kontrak kerja, cuti, jam kerja
- konsumen: produk rusak, refund, garansi, penipuan produk, hak pembeli

Identifikasi dan jawab HANYA dalam format JSON tanpa markdown:
{{"domain": "ketenagakerjaan/konsumen/lainnya", "is_clear": true/false, "clarification_question": "..." atau null}}"""),
    ("human", "{query}")
])

@traceable(name="node_1_clarify_context")
def clarify_context(state: AgentState) -> AgentState:
    """Node 1: Identifikasi domain hukum dan validasi kejelasan pertanyaan."""
    query  = state["user_query"]
    chain  = CLARIFY_PROMPT | llm
    result = chain.invoke({"query": query})

    try:
        raw    = result.content.strip().strip("```json").strip("```").strip()
        parsed = json.loads(raw)
    except Exception:
        parsed = {"domain": "lainnya", "is_clear": True, "clarification_question": None}

    needs_clarification = not parsed.get("is_clear", True)
    clarif_response     = None
    if needs_clarification:
        clarif_response = parsed.get("clarification_question",
                                     "Bisa ceritakan lebih detail situasi Anda?")

    return {
        **state,
        "legal_domain"        : parsed.get("domain", "lainnya"),
        "clarified_query"     : query,
        "needs_clarification" : needs_clarification,
        "final_response"      : clarif_response,
    }

# ── Test Node 1
print("Test Node 1 — clarify_context\n")
test_state = {
    "user_query": "Saya di-PHK tapi tidak dapat pesangon, apa hak saya?",
    "conversation_history": [], "legal_domain": None,
    "clarified_query": None, "needs_clarification": False,
    "retrieved_docs": [], "source_references": [], "relevance_scores": [],
    "answer": None, "cited_articles": [], "confidence_level": 0.0,
    "action_steps": [], "final_response": None,
}
r1 = clarify_context(test_state)
print(f"Domain terdeteksi   : {r1['legal_domain']}")
print(f"Perlu klarifikasi   : {r1['needs_clarification']}")
print(f"Query yang diproses : {r1['clarified_query']}")


Test Node 1 — clarify_context

Domain terdeteksi   : ketenagakerjaan
Perlu klarifikasi   : False
Query yang diproses : Saya di-PHK tapi tidak dapat pesangon, apa hak saya?


---
# 9. LangGraph Node 2 — retrieve_documents

Node kedua bertugas mencari dokumen yang paling relevan dengan pertanyaan dari ChromaDB. Ini adalah implementasi dari tahap "Retrieval" dalam RAG.

Cara kerjanya: pertanyaan diubah menjadi vektor menggunakan model embedding yang sama dengan yang dipakai saat mengindeks dokumen, kemudian sistem mencari chunk-chunk dengan vektor paling mirip menggunakan cosine similarity.

Ada dua hal penting yang dilakukan di sini. Pertama, sistem mengambil 10 hasil teratas dari seluruh dokumen lebih dahulu, kemudian menyaringnya secara manual berdasarkan domain yang sudah diidentifikasi node sebelumnya. Ini memastikan dokumen yang dikembalikan benar-benar berasal dari UU yang sesuai topik pertanyaan.

Kedua, skor similarity yang dikembalikan ChromaDB dinormalisasi ke rentang 0 sampai 1. Normalisasi ini penting agar nilai `confidence_level` di node berikutnya bisa bermakna dan tidak bernilai 0 akibat perbedaan skala pengukuran antar versi ChromaDB.


In [ ]:
# Node 2: Pencarian dokumen relevan dari ChromaDB dengan filter domain
@traceable(name="node_2_retrieve_documents")
def retrieve_documents(state: AgentState) -> AgentState:
    """Node 2: Ambil dokumen hukum relevan dari Chroma vector store."""
    query  = state["clarified_query"] or state["user_query"]
    domain = state.get("legal_domain")

    # Ambil lebih banyak dulu, filter manual setelahnya
    docs_scores_all = vectorstore.similarity_search_with_relevance_scores(query, k=10)

    # Filter manual berdasarkan domain
    if domain and domain != "lainnya":
        docs_scores = [
            (doc, score) for doc, score in docs_scores_all
            if doc.metadata.get("domain") == domain
        ]
        # Fallback: kalau hasil filter kosong, pakai semua
        if not docs_scores:
            docs_scores = docs_scores_all
    else:
        docs_scores = docs_scores_all

    # Ambil top 4 saja
    docs_scores = docs_scores[:4]

    docs    = [d for d, _ in docs_scores]
    scores  = [float(s) for _, s in docs_scores]

    # Normalisasi skor ke rentang 0–1
    if scores:
        min_s, max_s = min(scores), max(scores)
        if max_s > min_s:
            scores_norm = [(s - min_s) / (max_s - min_s) for s in scores]
        else:
            scores_norm = [1.0 for _ in scores]
    else:
        scores_norm = []

    sources = [d.metadata.get("source", "-") for d in docs]

    print(f"  [retrieve] domain filter: {domain}")
    print(f"  [retrieve] docs ditemukan: {len(docs)}")
    print(f"  [retrieve] scores norm: {[round(s,3) for s in scores_norm]}")

    return {
        **state,
        "retrieved_docs"   : docs,
        "source_references": sources,
        "relevance_scores" : scores_norm,  # pakai skor yang sudah dinormalisasi
    }

# ── Test Node 2
print("Test Node 2 — retrieve_documents\n")
r2 = retrieve_documents(r1)
print(f"Dokumen ditemukan: {len(r2['retrieved_docs'])}")
for i, (doc, score) in enumerate(zip(r2["retrieved_docs"], r2["relevance_scores"]), 1):
    print(f"\n  [{i}] Sumber : {doc.metadata.get('source', '-')}")
    print(f"       Score  : {score:.4f}")
    print(f"       Isi    : {doc.page_content[:200]}...")


Test Node 2 — retrieve_documents

  [retrieve] domain filter: ketenagakerjaan
  [retrieve] docs ditemukan: 4
  [retrieve] scores norm: [1.0, 1.0, 0.0, 0.0]
Dokumen ditemukan: 4

  [1] Sumber : uu_ketenagakerjaan.pdf
       Score  : 1.0000
       Isi    : Pasal 156
(1) Dalam hal terjadi pemutusan hubungan kerja, pengusaha diwajibkan membayar uang pesangon
dan atau uang penghargaan masa kerja dan uang penggantian hak yang seharusnya diterima.
(2) Perhit...

  [2] Sumber : uu_ketenagakerjaan.pdf
       Score  : 1.0000
       Isi    : Pasal 156
(1) Dalam hal terjadi pemutusan hubungan kerja, pengusaha diwajibkan membayar uang pesangon
dan atau uang penghargaan masa kerja dan uang penggantian hak yang seharusnya diterima.
(2) Perhit...

  [3] Sumber : uu_ketenagakerjaan.pdf
       Score  : 0.0003
       Isi    : h. masa kerja 24 (dua puluh empat) tahun atau lebih, 10 (sepuluh ) bulan upah.
(4) Uang penggantian hak yang seharusnya diterima sebagaimana dimaksud dalam ayat (1) meliputi :
a. cu

---
## 9a. Pengecekan Skor Mentah ChromaDB

Cell ini digunakan untuk mengecek nilai skor asli yang dikembalikan ChromaDB sebelum dinormalisasi. Berguna untuk memahami rentang nilai yang dihasilkan dan memastikan proses normalisasi di node 2 bekerja seperti yang diharapkan.


In [ ]:
# Cek skor mentah dari ChromaDB sebelum normalisasi
query_test = "Saya di-PHK mendadak tanpa pesangon, apa yang harus saya lakukan?"

docs_scores = vectorstore.similarity_search_with_relevance_scores(query_test, k=4)

print(f"Jumlah hasil: {len(docs_scores)}")
for i, (doc, score) in enumerate(docs_scores, 1):
    print(f"[{i}] Score raw: {score} | Domain: {doc.metadata.get('domain')}")
    print(f"     Isi: {doc.page_content[:100]}...")

Jumlah hasil: 4
[1] Score raw: 0.11236941580310611 | Domain: ketenagakerjaan
     Isi: Pasal 156
(1) Dalam hal terjadi pemutusan hubungan kerja, pengusaha diwajibkan membayar uang pesango...
[2] Score raw: 0.11236941580310611 | Domain: ketenagakerjaan
     Isi: Pasal 156
(1) Dalam hal terjadi pemutusan hubungan kerja, pengusaha diwajibkan membayar uang pesango...
[3] Score raw: 0.10841941316066517 | Domain: konsumen
     Isi: e. melakukan pemeriksaan di tempat tertentu yang di duga terdapat bahan bukti serta 
melakukan penyi...
[4] Score raw: 0.10841941316066517 | Domain: konsumen
     Isi: e. melakukan pemeriksaan di tempat tertentu yang di duga terdapat bahan bukti serta 
melakukan penyi...


---
# 10. LangGraph Node 3 — analyze_and_answer

Node ketiga adalah inti dari seluruh sistem. Di sini model bahasa menerima pertanyaan user beserta potongan-potongan dokumen UU yang sudah ditemukan, lalu menyusun jawaban yang mudah dipahami masyarakat awam.

Prompt yang digunakan dirancang dengan aturan yang ketat: model harus menjawab hanya berdasarkan konteks dokumen yang diberikan dan tidak boleh mengarang, wajib menyebut nomor pasal sebagai dasar hukum, serta selalu menyertakan catatan bahwa informasi ini bukan konsultasi hukum resmi.

Nilai `confidence_level` dihitung dari rata-rata skor relevance dokumen yang digunakan. Semakin tinggi nilainya, semakin yakin sistem bahwa jawaban didasarkan pada dokumen yang tepat dan relevan.

Jika jawaban yang dihasilkan menyebut informasi yang tidak ada dalam potongan dokumen yang diberikan sebagai konteks, itu tanda bahwa model mulai mengarang sendiri di luar dokumen.


In [ ]:
# Node 3: Analisis dokumen dan penyusunan jawaban dalam bahasa awam
ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """\
Kamu adalah asisten hukum yang membantu masyarakat awam Indonesia memahami hak-hak mereka.

ATURAN PENTING:
1. Gunakan bahasa Indonesia yang sederhana dan mudah dipahami
2. Hindari jargon hukum; jika terpaksa, jelaskan artinya
3. Jawab HANYA berdasarkan dokumen konteks yang diberikan
4. Selalu sebutkan dasar hukum (nama UU dan nomor pasal)
5. Tambahkan disclaimer bahwa ini bukan konsultasi hukum resmi

Format jawaban wajib:
📌 Penjelasan singkat (2-3 kalimat)
⚖️ Dasar hukum: [UU dan Pasal]
✅ Hak atau langkah yang bisa dilakukan
⚠️ Disclaimer"""),
    ("human", """Dokumen hukum relevan:
{context}

---
Pertanyaan: {query}""")
])

@traceable(name="node_3_analyze_and_answer")
def analyze_and_answer(state: AgentState) -> AgentState:
    """Node 3: Analisis dokumen dan formulasi jawaban dalam bahasa awam."""
    query  = state["clarified_query"]
    docs   = state["retrieved_docs"]
    scores = state["relevance_scores"]

    # Format konteks dari dokumen
    context = "\n\n".join([
        f"[{doc.metadata.get('source', 'Sumber tidak diketahui')}]\n{doc.page_content}"
        for doc in docs
    ])

    chain  = ANSWER_PROMPT | llm
    result = chain.invoke({"context": context, "query": query})

    # Ekstrak referensi pasal
    citations = list(set(re.findall(
        r"(?:UU|Pasal|Undang-Undang)[\w\s\.\-/,]+(?:No\.?\s*\d+)?(?:/\d+)?",
        result.content
    )))[:5]

    confidence = round(sum(scores) / len(scores), 2) if scores else 0.0

    return {
        **state,
        "answer"          : result.content,
        "cited_articles"  : citations,
        "confidence_level": confidence,
    }

# ── Test Node 3
print("Test Node 3 — analyze_and_answer\n")
r3 = analyze_and_answer(r2)
print("JAWABAN LLM:")
print("=" * 60)
print(r3["answer"])
print("=" * 60)
print(f"\nConfidence level : {r3['confidence_level']}")
print(f"Pasal dikutip    : {r3['cited_articles']}")


Test Node 3 — analyze_and_answer

JAWABAN LLM:
📌 Penjelasan singkat  
Jika Anda di-PHK (Pemutusan Hubungan Kerja), pengusaha wajib membayar uang pesangon, uang penghargaan masa kerja, dan uang penggantian hak yang belum Anda terima. Besaran uang pesangon tergantung lama masa kerja Anda, dan Anda juga berhak atas penggantian hak seperti cuti tahunan yang belum diambil.

⚖️ Dasar hukum:  
Undang-Undang Ketenagakerjaan, Pasal 156 ayat (1), (2), dan (4)

✅ Hak atau langkah yang bisa dilakukan:  
- Anda berhak menuntut pembayaran uang pesangon sesuai masa kerja Anda (misal, masa kerja 1 tahun dapat 2 bulan upah).  
- Anda juga berhak atas uang penghargaan masa kerja dan uang penggantian hak seperti cuti tahunan yang belum diambil, ongkos pulang, dan penggantian perumahan serta pengobatan jika memenuhi syarat.  
- Jika pengusaha tidak membayar, Anda dapat mengajukan pengaduan ke Dinas Ketenagakerjaan setempat atau menempuh jalur hukum untuk menuntut hak Anda.

⚠️ Disclaimer  
Penjelasan ini 

---
# 11. LangGraph Node 4 — recommend_action

Node terakhir melengkapi respons yang sudah dibuat node sebelumnya dengan menambahkan informasi tindak lanjut yang konkret.

Berdasarkan domain hukum yang terdeteksi di awal pipeline, sistem memilih daftar instansi yang relevan untuk dihubungi. Pertanyaan ketenagakerjaan diarahkan ke Disnaker atau Pengadilan Hubungan Industrial. Pertanyaan konsumen diarahkan ke BPSK atau YLKI.

Selain itu, sistem selalu menyertakan informasi bantuan hukum gratis yang bisa diakses siapa saja, seperti Posbakum di Pengadilan Negeri dan YLBHI.

Jika nilai `confidence_level` dari node sebelumnya berada di bawah 0.65, sistem secara otomatis menambahkan catatan yang mendorong user untuk berkonsultasi langsung dengan pengacara atau LBH guna mendapatkan jawaban yang lebih akurat.


In [ ]:
# Node 4: Tambahkan rekomendasi instansi dan informasi tindak lanjut
AGENCY_MAP = {
    "ketenagakerjaan": [
        "Dinas Tenaga Kerja (Disnaker) kabupaten/kota setempat",
        "Pengadilan Hubungan Industrial (PHI) untuk sengketa PHK",
        "BPJS Ketenagakerjaan untuk klaim jaminan sosial",
    ],
    "konsumen": [
        "Badan Penyelesaian Sengketa Konsumen (BPSK) setempat",
        "YLKI — Yayasan Lembaga Konsumen Indonesia",
        "OJK (1500-655) untuk produk/jasa keuangan",
    ],
}

FREE_AID = [
    "Pos Bantuan Hukum (Posbakum) di Pengadilan Negeri — GRATIS",
    "YLBHI (Yayasan LBH Indonesia): 021-3929840",
]

@traceable(name="node_4_recommend_action")
def recommend_action(state: AgentState) -> AgentState:
    """Node 4: Tambahkan rekomendasi tindak lanjut ke respons akhir."""
    domain     = state.get("legal_domain", "lainnya")
    answer     = state.get("answer", "")
    confidence = state.get("confidence_level", 0.0)
    sources    = state.get("source_references", [])

    agencies = AGENCY_MAP.get(domain, [
        "LBH (Lembaga Bantuan Hukum) terdekat",
        "Pengadilan Negeri setempat",
    ])

    # Format sumber hukum
    source_text = ""
    if sources:
        unique = list(dict.fromkeys(sources))[:2]
        source_text  = "\n\n---\n📚 **Sumber dokumen yang digunakan:**\n"
        source_text += "\n".join([f"- {s}" for s in unique])

    # Format instansi tujuan
    agency_text  = "\n\n🏛️ **Langkah selanjutnya — hubungi:**\n"
    agency_text += "\n".join([f"- {a}" for a in agencies])

    # Bantuan hukum gratis
    free_text  = "\n\n🆓 **Bantuan hukum gratis:**\n"
    free_text += "\n".join([f"- {a}" for a in FREE_AID])

    # Disclaimer ekstra jika confidence rendah
    extra = ""
    if confidence < 0.65:
        extra = (
            "\n\n⚠️ **Catatan:** Relevansi dokumen rendah. Sangat disarankan "
            "berkonsultasi langsung dengan LBH atau pengacara."
        )

    final = answer + source_text + agency_text + free_text + extra

    return {
        **state,
        "action_steps"  : agencies,
        "final_response": final,
    }

# ── Test Node 4
print("Test Node 4 — recommend_action\n")
r4 = recommend_action(r3)
print("RESPONS FINAL LENGKAP:")
print("=" * 60)
print(r4["final_response"])


Test Node 4 — recommend_action

RESPONS FINAL LENGKAP:
📌 Penjelasan singkat  
Jika Anda di-PHK (Pemutusan Hubungan Kerja), pengusaha wajib membayar uang pesangon, uang penghargaan masa kerja, dan uang penggantian hak yang belum Anda terima. Besaran uang pesangon tergantung lama masa kerja Anda, dan Anda juga berhak atas penggantian hak seperti cuti tahunan yang belum diambil.

⚖️ Dasar hukum:  
Undang-Undang Ketenagakerjaan, Pasal 156 ayat (1), (2), dan (4)

✅ Hak atau langkah yang bisa dilakukan:  
- Anda berhak menuntut pembayaran uang pesangon sesuai masa kerja Anda (misal, masa kerja 1 tahun dapat 2 bulan upah).  
- Anda juga berhak atas uang penghargaan masa kerja dan uang penggantian hak seperti cuti tahunan yang belum diambil, ongkos pulang, dan penggantian perumahan serta pengobatan jika memenuhi syarat.  
- Jika pengusaha tidak membayar, Anda dapat mengajukan pengaduan ke Dinas Ketenagakerjaan setempat atau menempuh jalur hukum untuk menuntut hak Anda.

⚠️ Disclaimer  
Penjela

---
# 12. LangGraph — Merangkai State Machine

Setelah semua node siap, kita rangkai semuanya menjadi satu state machine menggunakan `StateGraph`. Di sinilah LangGraph bekerja: mengatur urutan eksekusi node dan menangani percabangan berdasarkan kondisi state saat itu.

Ada dua conditional edge yang menentukan alur:

Pertama, `should_clarify` yang berjalan setelah node clarify. Jika pertanyaan perlu klarifikasi, alur berhenti dan pertanyaan balik dikembalikan ke user. Jika tidak, alur lanjut ke node retrieve.

Kedua, `has_relevant_docs` yang berjalan setelah node retrieve. Jika skor tertinggi dokumen yang ditemukan di bawah 0.4, berarti pertanyaan di luar cakupan dokumen dan alur diarahkan ke node `out_of_scope`. Jika skor cukup, alur lanjut ke node analyze untuk menyusun jawaban.

Setelah semua node dan edge didaftarkan, kita compile graph agar siap dijalankan.


In [ ]:
# Merangkai semua node menjadi state machine LangGraph
def should_clarify(state: AgentState) -> str:
    """Conditional edge: apakah perlu klarifikasi?"""
    return "ask_user" if state.get("needs_clarification") else "retrieve"

def has_relevant_docs(state: AgentState) -> str:
    """Conditional edge: apakah dokumen cukup relevan?"""
    scores = state.get("relevance_scores", [])
    if not scores or max(scores) < 0.4:
        return "out_of_scope"
    return "analyze"

def handle_out_of_scope(state: AgentState) -> AgentState:
    """Tangani pertanyaan di luar cakupan dokumen."""
    return {
        **state,
        "final_response": (
            "Maaf, pertanyaan Anda berada di luar cakupan dokumen yang tersedia.\n\n"
            "Sistem ini hanya mencakup:\n"
            "- ⚖️ Hukum Ketenagakerjaan (PHK, pesangon, upah)\n"
            "- 🛒 Perlindungan Konsumen (hak pembeli, refund, garansi)\n\n"
            "Untuk pertanyaan hukum lainnya, silakan hubungi:\n"
            "- LBH (Lembaga Bantuan Hukum) terdekat\n"
            "- YLBHI: 021-3929840\n"
            "- Posbakum di Pengadilan Negeri (gratis)"
        )
    }

# Bangun graph
workflow = StateGraph(AgentState)

workflow.add_node("clarify",      clarify_context)
workflow.add_node("retrieve",     retrieve_documents)
workflow.add_node("analyze",      analyze_and_answer)
workflow.add_node("recommend",    recommend_action)
workflow.add_node("out_of_scope", handle_out_of_scope)

workflow.set_entry_point("clarify")

workflow.add_conditional_edges(
    "clarify",
    should_clarify,
    {"ask_user": END, "retrieve": "retrieve"}
)

workflow.add_conditional_edges(
    "retrieve",
    has_relevant_docs,
    {"out_of_scope": "out_of_scope", "analyze": "analyze"}
)

workflow.add_edge("analyze",      "recommend")
workflow.add_edge("recommend",    END)
workflow.add_edge("out_of_scope", END)

# Compile
app = workflow.compile()

print("LangGraph state machine berhasil dibangun!")
print()
print("Alur eksekusi:")
print("  clarify_context")
print("    ├── needs_clarification=True  → END (tanya balik user)")
print("    └── needs_clarification=False → retrieve_documents")
print("                                       ├── score < 0.4 → out_of_scope → END")
print("                                       └── score ≥ 0.4 → analyze_and_answer")
print("                                                             └── recommend_action → END")


LangGraph state machine berhasil dibangun!

Alur eksekusi:
  clarify_context
    ├── needs_clarification=True  → END (tanya balik user)
    └── needs_clarification=False → retrieve_documents
                                       ├── score < 0.4 → out_of_scope → END
                                       └── score ≥ 0.4 → analyze_and_answer
                                                             └── recommend_action → END


---
# 13. Fungsi Utama — ask_chatbot_hukum

Fungsi ini adalah pintu masuk utama untuk mengirim pertanyaan ke sistem. Cara kerjanya adalah menyiapkan state awal dengan semua field kosong atau bernilai default, kemudian menyerahkannya ke LangGraph untuk diproses melalui seluruh pipeline dari node pertama hingga terakhir.

Decorator `@traceable` di atas fungsi ini memastikan bahwa setiap kali fungsi dipanggil, seluruh rangkaian eksekusi dari node clarify hingga recommend tercatat sebagai satu trace utuh di LangSmith, termasuk semua node di dalamnya sebagai sub-trace.

Nilai `confidence_level` di output menunjukkan tingkat keyakinan sistem bahwa jawaban yang diberikan didasarkan pada dokumen yang tepat. Nilai di atas 0.65 menandakan dokumen yang ditemukan sangat relevan dengan pertanyaan.


In [ ]:
# Fungsi utama untuk menjalankan seluruh pipeline chatbot
@traceable(name="chatbot_hukum_pipeline")
def ask_chatbot_hukum(pertanyaan: str, history: List = None) -> dict:
    """
    Fungsi utama Chatbot Bantuan Hukum.
    Semua langkah otomatis di-trace oleh LangSmith.
    """
    initial_state = {
        "user_query"           : pertanyaan,
        "conversation_history" : history or [],
        "legal_domain"         : None,
        "clarified_query"      : None,
        "needs_clarification"  : False,
        "retrieved_docs"       : [],
        "source_references"    : [],
        "relevance_scores"     : [],
        "answer"               : None,
        "cited_articles"       : [],
        "confidence_level"     : 0.0,
        "action_steps"         : [],
        "final_response"       : None,
    }

    return app.invoke(initial_state)

# ── Test fungsi utama
print("=" * 65)
pertanyaan_demo = "Saya di-PHK mendadak tanpa pesangon, apa yang harus saya lakukan?"
print(f"Pertanyaan: {pertanyaan_demo}")
print("=" * 65)

hasil = ask_chatbot_hukum(pertanyaan_demo)

print(f"Domain hukum     : {hasil['legal_domain']}")
print(f"Confidence level : {hasil['confidence_level']}")
print()
print("RESPONS FINAL:")
print(hasil["final_response"])


Pertanyaan: Saya di-PHK mendadak tanpa pesangon, apa yang harus saya lakukan?
  [retrieve] domain filter: ketenagakerjaan
  [retrieve] docs ditemukan: 2
  [retrieve] scores norm: [1.0, 1.0]
Domain hukum     : ketenagakerjaan
Confidence level : 1.0

RESPONS FINAL:
📌 Jika Anda di-PHK (Pemutusan Hubungan Kerja) mendadak tanpa mendapatkan uang pesangon, pengusaha sebenarnya wajib membayar uang pesangon sesuai masa kerja Anda. Besaran pesangon minimal diatur berdasarkan lamanya Anda bekerja.

⚖️ Dasar hukum: Undang-Undang Ketenagakerjaan, Pasal 156 ayat (1) dan (2)

✅ Langkah yang bisa Anda lakukan:
1. Minta penjelasan dan bukti tertulis dari pengusaha mengenai alasan tidak dibayarnya pesangon.
2. Hitung hak pesangon Anda sesuai masa kerja (misalnya, masa kerja 2 tahun berarti berhak atas 3 bulan upah).
3. Ajukan mediasi atau pengaduan ke Dinas Ketenagakerjaan setempat untuk menuntut hak pesangon Anda.
4. Jika perlu, konsultasikan dengan pengacara atau lembaga bantuan hukum untuk pendamping

---
# 14. LangSmith — Melihat Hasil Tracing

Di sini kita ambil data pencatatan yang sudah dilakukan LangSmith selama pipeline dijalankan. Setiap kali `ask_chatbot_hukum` dipanggil, LangSmith secara otomatis mencatat seluruh langkah eksekusi mulai dari node clarify hingga recommend, beserta input, output, dan waktu yang dihabiskan masing-masing node.

Kolom `Latency` menunjukkan berapa lama tiap node memerlukan waktu untuk berjalan. Node `analyze_and_answer` biasanya memiliki latency paling tinggi karena di sanalah model bahasa dipanggil dengan konteks dokumen yang paling panjang.

Selain dari notebook ini, kita juga bisa melihat semua trace secara visual di dashboard LangSmith, termasuk pohon eksekusi tiap node beserta input dan output lengkapnya.


In [ ]:
# Mengambil dan menampilkan data tracing dari LangSmith
ls_client = LangSmithClient()

try:
    runs = list(ls_client.list_runs(
        project_name="uas-chatbot-hukum",
        limit=8
    ))

    if runs:
        print(f"Berhasil mengambil {len(runs)} run terbaru dari LangSmith:\n")
        run_data = []
        for run in runs:
            latency = (
                (run.end_time - run.start_time).total_seconds()
                if run.end_time and run.start_time else None
            )
            run_data.append({
                "Nama Node" : run.name,
                "Status"    : run.status,
                "Latency"   : f"{latency:.2f}s" if latency else "N/A",
            })

        df_runs = pd.DataFrame(run_data)
        print(df_runs.to_string(index=False))
    else:
        print("Belum ada run tercatat. Jalankan ask_chatbot_hukum() terlebih dahulu.")

except Exception as e:
    print(f"Tidak bisa mengambil data LangSmith: {e}")
    print("Pastikan LANGCHAIN_API_KEY sudah diset dengan benar.")

print()
print("Dashboard LangSmith : https://smith.langchain.com")
print("Project             : uas-chatbot-hukum")


Berhasil mengambil 8 run terbaru dari LangSmith:

                Nama Node  Status Latency
node_2_retrieve_documents success   0.61s
                 retrieve success   0.61s
           should_clarify success   0.00s
               ChatOpenAI success  27.41s
       ChatPromptTemplate success   0.00s
         RunnableSequence success  27.41s
   node_1_clarify_context success  27.41s
                  clarify success  27.41s

Dashboard LangSmith : https://smith.langchain.com
Project             : uas-chatbot-hukum


---
# 15. Demo — Berbagai Pertanyaan dari Dua Domain

Pada bagian ini kita uji sistem dengan empat pertanyaan yang mencakup kedua domain yang didukung. Dua pertanyaan pertama seputar ketenagakerjaan, dua pertanyaan berikutnya seputar perlindungan konsumen.

Tujuannya adalah memvalidasi bahwa pipeline bekerja dari ujung ke ujung dengan benar, dan bahwa setiap pertanyaan mendapat jawaban dari dokumen yang sesuai domainnya. Perhatikan kolom `domain` dan `top_source` pada tabel ringkasan di bagian bawah, keduanya harus selaras satu sama lain.


In [ ]:
# Uji coba sistem dengan pertanyaan dari dua domain berbeda
demo_questions = [
    "Berapa pesangon yang saya dapatkan jika di-PHK setelah bekerja 6 tahun?",
    "Apakah kontrak kerja lisan memiliki kekuatan hukum?",
    "Toko online tidak mau refund padahal produk yang dikirim rusak, apa hak saya?",
    "Saya membeli barang tapi tidak sesuai deskripsi, bolehkah saya minta ganti rugi?",
]

hasil_demo = []

for i, pertanyaan in enumerate(demo_questions, 1):
    print(f"{'='*65}")
    print(f"[{i}] {pertanyaan}")
    print(f"{'='*65}")

    hasil = ask_chatbot_hukum(pertanyaan)

    print(f"Domain    : {hasil['legal_domain']}")
    print(f"Confidence: {hasil['confidence_level']}")
    print(f"\nJawaban:")
    print(hasil["final_response"][:500] + "...")
    print()

    hasil_demo.append({
        "pertanyaan" : pertanyaan,
        "domain"     : hasil["legal_domain"],
        "confidence" : hasil["confidence_level"],
        "top_source" : hasil["source_references"][0] if hasil["source_references"] else "-",
        "jawaban"    : hasil["final_response"][:200] + "...",
    })

print("\nRingkasan hasil demo:")
df_demo = pd.DataFrame(hasil_demo)
df_demo[["pertanyaan", "domain", "confidence", "top_source"]]


[1] Berapa pesangon yang saya dapatkan jika di-PHK setelah bekerja 6 tahun?
  [retrieve] domain filter: ketenagakerjaan
  [retrieve] docs ditemukan: 4
  [retrieve] scores norm: [1.0, 0.996, 0.0, 0.0]
Domain    : ketenagakerjaan
Confidence: 0.5

Jawaban:
📌 Penjelasan singkat:  
Jika Anda di-PHK setelah bekerja selama 6 tahun, Anda berhak mendapatkan uang pesangon minimal sebesar 6 bulan upah. Hal ini sesuai dengan ketentuan perhitungan uang pesangon berdasarkan masa kerja.

⚖️ Dasar hukum:  
Undang-Undang Ketenagakerjaan, Pasal 156 ayat (2) huruf f, yang menyatakan bahwa untuk masa kerja 5 tahun atau lebih tetapi kurang dari 6 tahun, pesangon minimal adalah 6 bulan upah.

✅ Hak atau langkah yang bisa dilakukan:  
- Anda berhak menerima uang pesa...

[2] Apakah kontrak kerja lisan memiliki kekuatan hukum?
  [retrieve] domain filter: ketenagakerjaan
  [retrieve] docs ditemukan: 4
  [retrieve] scores norm: [1.0, 1.0, 0.0, 0.0]
Domain    : ketenagakerjaan
Confidence: 0.5

Jawaban:
📌 Penjela

,pertanyaan,domain,confidence,top_source
0,Berapa pesangon yang saya dapatkan jika di-PHK...,ketenagakerjaan,0.50,uu_ketenagakerjaan.pdf
1,Apakah kontrak kerja lisan memiliki kekuatan h...,ketenagakerjaan,0.50,uu_ketenagakerjaan.pdf
2,Toko online tidak mau refund padahal produk ya...,konsumen,0.63,uu_perlindungan_konsumen.pdf
3,Saya membeli barang tapi tidak sesuai deskrips...,konsumen,0.50,uu_perlindungan_konsumen.pdf


---
# 16. Visualisasi — Distribusi Chunk per Domain

Grafik ini menampilkan jumlah chunk yang ada di vector store untuk masing-masing domain. Distribusi ini penting karena mencerminkan seberapa banyak materi referensi yang tersedia untuk tiap domain hukum.

Domain dengan lebih banyak chunk cenderung menghasilkan retrieval yang lebih baik karena sistem punya lebih banyak pilihan potongan teks untuk dicocokkan dengan pertanyaan. Jika selisihnya terlalu jauh antara dua domain, domain dengan lebih sedikit chunk mungkin perlu tambahan dokumen referensi.


In [ ]:
# Visualisasi distribusi jumlah chunk per domain di vector store
domain_count = {}
for chunk in all_chunks:
    d = chunk.metadata.get("domain", "unknown")
    domain_count[d] = domain_count.get(d, 0) + 1

df_dist = pd.DataFrame([
    {"domain": k, "jumlah_chunks": v}
    for k, v in domain_count.items()
])

fig = px.bar(
    df_dist,
    x="domain",
    y="jumlah_chunks",
    color="domain",
    text="jumlah_chunks",
    title="Distribusi Chunks Dokumen Hukum per Domain di Vector Store",
    labels={"domain": "Domain Hukum", "jumlah_chunks": "Jumlah Chunks"},
    color_discrete_sequence=["#185FA5", "#0F6E56"]
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, height=420, width=650)
fig.show()

print("\nDetail distribusi:")
print(df_dist.to_string(index=False))



Detail distribusi:
         domain  jumlah_chunks
ketenagakerjaan            229
       konsumen             81


---
# 17. Visualisasi — Posisi Embedding Dokumen dalam 2D

Setiap chunk dokumen direpresentasikan sebagai vektor berdimensi 1536. Agar bisa divisualisasikan, kita gunakan PCA (Principal Component Analysis) untuk mereduksi dimensinya menjadi hanya 2, lalu ditampilkan sebagai scatter plot interaktif.

Setiap titik pada grafik mewakili satu chunk dokumen. Warna titik menunjukkan domain asal chunk tersebut. Titik-titik yang berdekatan memiliki makna semantik yang serupa.

Jika chunk dari domain yang sama cenderung berkumpul di area yang berdekatan dan terpisah dari domain lain, itu menunjukkan bahwa model embedding berhasil menangkap perbedaan tema antara UU Ketenagakerjaan dan UU Perlindungan Konsumen.


In [ ]:
# Visualisasi posisi embedding dokumen menggunakan reduksi dimensi PCA 2D
chroma_data = vectorstore.get(include=["documents", "metadatas", "embeddings"])

if chroma_data and len(chroma_data.get("embeddings", [])) > 0:
    emb_array = np.array(chroma_data["embeddings"])
    meta_list = chroma_data["metadatas"]
    text_list = chroma_data["documents"]

    pca    = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(emb_array)

    df_viz = pd.DataFrame({
        "x"      : coords[:, 0],
        "y"      : coords[:, 1],
        "domain" : [m.get("domain", "-") for m in meta_list],
        "sumber" : [m.get("source", "-") for m in meta_list],
        "teks"   : [t[:120] + "..." for t in text_list],
    })

    fig = px.scatter(
        df_viz,
        x="x", y="y",
        color="domain",
        hover_data=["sumber", "teks"],
        title="Visualisasi Embedding Chunks Dokumen Hukum (PCA 2D)",
        labels={"x": "Komponen PCA 1", "y": "Komponen PCA 2"},
        color_discrete_sequence=["#185FA5", "#0F6E56"]
    )
    fig.update_traces(marker=dict(size=8, opacity=0.8))
    fig.update_layout(height=520, width=800)
    fig.show()

    print(f"Total titik divisualisasikan: {len(df_viz)}")
    print(f"Variansi yang dijelaskan PCA : {pca.explained_variance_ratio_.sum():.2%}")
else:
    print("Embeddings tidak tersedia dari vector store.")


Total titik divisualisasikan: 620
Variansi yang dijelaskan PCA : 16.50%


---
# 18. Evaluasi — Akurasi Identifikasi Domain

Pada bagian ini kita evaluasi seberapa akurat sistem dalam mengidentifikasi domain hukum dari berbagai pertanyaan. Setiap query uji sudah diketahui domain yang seharusnya, kemudian kita bandingkan dengan domain yang terdeteksi oleh sistem.

Kolom `is_correct` menunjukkan apakah domain yang terdeteksi sesuai dengan yang diharapkan. Akurasi di atas 80% menunjukkan bahwa sistem cukup andal dalam memahami dan mengklasifikasikan pertanyaan hukum.

Jika akurasi rendah, kemungkinan besar perlu ada penyesuaian pada prompt di node `clarify_context`, misalnya dengan memperjelas deskripsi setiap domain atau menambah contoh pertanyaan.


In [ ]:
# Evaluasi akurasi identifikasi domain pada berbagai pertanyaan uji
eval_queries = [
    {"query": "Berapa pesangon PHK setelah kerja 5 tahun?",           "expected": "ketenagakerjaan"},
    {"query": "Apakah saya berhak cuti hamil dibayar?",               "expected": "ketenagakerjaan"},
    {"query": "Kontrak kerja saya habis, apa hak saya?",              "expected": "ketenagakerjaan"},
    {"query": "Produk yang saya beli rusak, boleh dikembalikan?",     "expected": "konsumen"},
    {"query": "Toko menolak refund, saya harus ke mana?",             "expected": "konsumen"},
    {"query": "Iklan produk menipu, apa yang bisa saya lakukan?",     "expected": "konsumen"},
]

eval_rows = []
print("Menjalankan evaluasi...\n")

for item in eval_queries:
    hasil = ask_chatbot_hukum(item["query"])
    retrieved = hasil.get("legal_domain", "lainnya")
    top_score = hasil["relevance_scores"][0] if hasil["relevance_scores"] else 0.0

    eval_rows.append({
        "query"            : item["query"][:55] + "...",
        "expected_domain"  : item["expected"],
        "retrieved_domain" : retrieved,
        "top_score"        : round(top_score, 3),
        "is_correct"       : item["expected"] == retrieved,
    })

df_eval   = pd.DataFrame(eval_rows)
akurasi   = df_eval["is_correct"].mean()

print(f"Akurasi identifikasi domain : {akurasi:.0%}")
print(f"Benar : {df_eval['is_correct'].sum()} dari {len(df_eval)} query\n")
df_eval


Menjalankan evaluasi...

  [retrieve] domain filter: ketenagakerjaan
  [retrieve] docs ditemukan: 4
  [retrieve] scores norm: [1.0, 0.998, 0.0, 0.0]
  [retrieve] domain filter: ketenagakerjaan
  [retrieve] docs ditemukan: 4
  [retrieve] scores norm: [1.0, 0.996, 0.002, 0.0]
  [retrieve] domain filter: ketenagakerjaan
  [retrieve] docs ditemukan: 4
  [retrieve] scores norm: [1.0, 1.0, 0.0, 0.0]
  [retrieve] domain filter: konsumen
  [retrieve] docs ditemukan: 4
  [retrieve] scores norm: [1.0, 0.999, 0.0, 0.0]
  [retrieve] domain filter: konsumen
  [retrieve] docs ditemukan: 4
  [retrieve] scores norm: [1.0, 1.0, 0.015, 0.0]
  [retrieve] domain filter: konsumen
  [retrieve] docs ditemukan: 4
  [retrieve] scores norm: [1.0, 1.0, 0.0, 0.0]
Akurasi identifikasi domain : 100%
Benar : 6 dari 6 query



,query,expected_domain,retrieved_domain,top_score,is_correct
0,Berapa pesangon PHK setelah kerja 5 tahun?...,ketenagakerjaan,ketenagakerjaan,1.0,True
1,Apakah saya berhak cuti hamil dibayar?...,ketenagakerjaan,ketenagakerjaan,1.0,True
2,"Kontrak kerja saya habis, apa hak saya?...",ketenagakerjaan,ketenagakerjaan,1.0,True
3,"Produk yang saya beli rusak, boleh dikembalika...",konsumen,konsumen,1.0,True
4,"Toko menolak refund, saya harus ke mana?...",konsumen,konsumen,1.0,True
5,"Iklan produk menipu, apa yang bisa saya lakuka...",konsumen,konsumen,1.0,True


---
# 19. LangSmith — Ringkasan Performa Pipeline

Setelah menjalankan banyak pertanyaan, kita tampilkan ringkasan performa keseluruhan pipeline berdasarkan data yang tercatat di LangSmith. Data ini mencakup jumlah run, status eksekusi, dan rata-rata latency per node.

Grafik batang memperlihatkan perbandingan waktu eksekusi antar node. Biasanya node `analyze_and_answer` memiliki latency tertinggi karena di sanalah model bahasa dipanggil dengan konteks yang paling panjang. Informasi ini berguna untuk mengidentifikasi bagian mana dari pipeline yang paling bisa dioptimalkan jika sistem dikembangkan lebih lanjut.


In [ ]:
# Ringkasan performa pipeline dari data yang tercatat di LangSmith
try:
    runs = list(ls_client.list_runs(
        project_name="uas-chatbot-hukum",
        run_type="chain",
        limit=50
    ))

    if runs:
        run_data = []
        for run in runs:
            lat = (
                (run.end_time - run.start_time).total_seconds()
                if run.end_time and run.start_time else None
            )
            run_data.append({
                "nama_node"     : run.name,
                "status"        : run.status,
                "latency_detik" : round(lat, 2) if lat else None,
            })

        df_runs = pd.DataFrame(run_data)

        print(f"Total run tercatat : {len(df_runs)}")
        avg_lat = df_runs["latency_detik"].dropna().mean()
        print(f"Rata-rata latency  : {avg_lat:.2f} detik")
        print()

        # Ringkasan per node
        summary = df_runs.groupby("nama_node").agg(
            jumlah_run=("latency_detik", "count"),
            avg_latency=("latency_detik", "mean")
        ).round(2).reset_index()

        print("Performa per node:")
        print(summary.to_string(index=False))

        # Visualisasi latency
        fig = px.bar(
            summary,
            x="nama_node",
            y="avg_latency",
            text="avg_latency",
            title="Rata-rata Latency per Node (LangSmith)",
            labels={"nama_node": "Node", "avg_latency": "Latency (detik)"},
            color="avg_latency",
            color_continuous_scale="Blues"
        )
        fig.update_traces(textposition="outside")
        fig.update_layout(height=420, width=700, showlegend=False)
        fig.show()
    else:
        print("Belum ada run. Jalankan ask_chatbot_hukum() terlebih dahulu.")

except Exception as e:
    print(f"Tidak bisa mengambil data: {e}")

print("\nDashboard LangSmith : https://smith.langchain.com")
print("Project             : uas-chatbot-hukum")


Total run tercatat : 50
Rata-rata latency  : 1.21 detik

Performa per node:
                nama_node  jumlah_run  avg_latency
                LangGraph           2         4.25
         RunnableSequence           7         1.75
                  analyze           3         3.35
   chatbot_hukum_pipeline           2         4.25
                  clarify           3         0.53
        has_relevant_docs           4         0.00
   node_1_clarify_context           3         0.53
node_2_retrieve_documents           4         0.21
node_3_analyze_and_answer           3         3.35
  node_4_recommend_action           3         0.00
                recommend           3         0.00
                 retrieve           4         0.21
           should_clarify           4         0.00



Dashboard LangSmith : https://smith.langchain.com
Project             : uas-chatbot-hukum


---
# 20. Demo Interaktif — Antarmuka Chatbot dengan Gradio

Bagian terakhir ini membuat antarmuka web yang bisa digunakan langsung untuk mencoba chatbot. Gradio dipilih karena mudah diintegrasikan dengan Colab dan bisa menghasilkan link publik yang dapat diakses dari luar Colab.

Antarmuka menampilkan empat bagian output: jawaban lengkap dari sistem, informasi domain dan tingkat relevansi dokumen, sumber dokumen yang dijadikan referensi, dan pasal-pasal yang dikutip dalam jawaban.

Parameter `share=True` menghasilkan link publik sementara yang aktif selama cell ini berjalan. Untuk menghentikan server Gradio, klik tombol stop pada cell ini.


In [ ]:
# Antarmuka chatbot interaktif menggunakan Gradio
DOMAIN_LABELS = {
    "ketenagakerjaan": "⚖️ Ketenagakerjaan",
    "konsumen"        : "🛒 Perlindungan Konsumen",
    "lainnya"         : "📋 Di luar cakupan",
}

def gradio_chat(pertanyaan: str) -> tuple:
    if not pertanyaan or not pertanyaan.strip():
        return "Silakan masukkan pertanyaan terlebih dahulu.", "", "", ""

    try:
        hasil = ask_chatbot_hukum(pertanyaan)

        domain_label = DOMAIN_LABELS.get(
            hasil.get("legal_domain", "lainnya"), "📋 Lainnya"
        )
        conf_pct  = int(hasil.get("confidence_level", 0) * 100)
        conf_icon = "🟢" if conf_pct >= 65 else "🟡" if conf_pct >= 40 else "🔴"
        meta_text = f"{domain_label}  |  {conf_icon} Relevansi dokumen: {conf_pct}%"

        sources = "\n".join([
            f"- {s}" for s in list(dict.fromkeys(hasil.get("source_references", [])))[:2]
        ]) or "Tidak ditemukan"

        pasal = ", ".join(hasil.get("cited_articles", [])[:3]) or "-"

        return (
            hasil.get("final_response", "Maaf, terjadi kesalahan."),
            meta_text,
            sources,
            pasal
        )

    except Exception as e:
        return f"Terjadi kesalahan: {str(e)}", "", "", ""


with gr.Blocks(title="Chatbot Bantuan Hukum") as demo:
    gr.Markdown("""
# ⚖️ Chatbot Bantuan Hukum Masyarakat Awam
**Powered by LangChain + LangGraph + LangSmith | Model: gpt-4.1-mini**

Ajukan pertanyaan seputar **hukum ketenagakerjaan** (PHK, pesangon, upah) atau **perlindungan konsumen** (refund, garansi, hak pembeli).

> ⚠️ *Informasi hukum umum — bukan pengganti konsultasi hukum profesional.*
    """)

    with gr.Row():
        with gr.Column(scale=2):
            input_box  = gr.Textbox(
                label="Pertanyaan Hukum",
                placeholder="Contoh: Saya di-PHK tanpa pesangon, apa hak saya?",
                lines=3
            )
            submit_btn = gr.Button("Tanya ⚖️", variant="primary")

            gr.Examples(
                examples=[
                    ["Berapa pesangon yang saya dapat jika di-PHK setelah 5 tahun kerja?"],
                    ["Apakah kontrak kerja lisan sah secara hukum?"],
                    ["Produk yang saya beli rusak tapi toko menolak refund"],
                    ["Iklan produk menipu, apa yang bisa saya lakukan?"],
                ],
                inputs=input_box
            )

        with gr.Column(scale=3):
            output_jawaban = gr.Textbox(label="Jawaban", lines=16)
            output_meta    = gr.Textbox(label="Domain & Relevansi", lines=1)
            output_sources = gr.Textbox(label="Sumber Dokumen", lines=2)
            output_pasal   = gr.Textbox(label="Pasal yang Dikutip", lines=1)

    submit_btn.click(
        fn=gradio_chat,
        inputs=input_box,
        outputs=[output_jawaban, output_meta, output_sources, output_pasal]
    )

demo.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cca0ea5ad2f111a3e6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
# 21. Kesimpulan

Pada proyek UAS ini telah dibangun sebuah Chatbot Bantuan Hukum Masyarakat Awam yang menggunakan tiga library wajib secara terintegrasi dalam satu sistem.

## Peran Setiap Library

| Library | Implementasi dalam Proyek |
|---|---|
| LangChain | PyPDFLoader memuat PDF UU resmi, RecursiveCharacterTextSplitter memecah teks per pasal, OpenAIEmbeddings mengubah teks menjadi vektor, ChromaDB menyimpan dan mencari dokumen, ChatPromptTemplate merancang prompt tiap node |
| LangGraph | StateGraph merangkai 4 node yaitu clarify_context, retrieve_documents, analyze_and_answer, dan recommend_action, dengan conditional edges berbasis state dan AgentState sebagai wadah data antar node |
| LangSmith | Decorator traceable pada setiap node mencatat eksekusi secara otomatis, LangSmithClient mengambil data run untuk analisis performa, dashboard smith.langchain.com menampilkan trace lengkap secara visual |

## Alur Kerja Sistem

1. User mengirim pertanyaan hukum
2. Node clarify_context mengidentifikasi domain dan memvalidasi kejelasan pertanyaan
3. Node retrieve_documents mencari pasal relevan dari ChromaDB berdasarkan domain
4. Node analyze_and_answer menyusun jawaban dalam bahasa awam berdasarkan isi dokumen UU
5. Node recommend_action melengkapi respons dengan rekomendasi instansi dan informasi bantuan gratis
6. Seluruh langkah dicatat otomatis oleh LangSmith untuk keperluan monitoring dan evaluasi

## Dokumen Hukum yang Digunakan

- UU No. 13 Tahun 2003 tentang Ketenagakerjaan
- UU No. 8 Tahun 1999 tentang Perlindungan Konsumen

---

Sistem ini dibuat untuk keperluan edukasi dan tidak menggantikan konsultasi hukum profesional. Untuk kepastian hukum atas situasi spesifik, disarankan menghubungi LBH atau pengacara terdaftar.
